In [ ]:
from pathlib import Path
from cava_llm_manager.pipelines.base import PipelineSpec
from cava_llm_manager.schemas.genomic.mutations import GenomicBatchResult
from cava_llm_manager import get_registry
from cava_llm_manager.pipelines.pipeline_runner import run_pipeline

In [ ]:
reg = get_registry()
print("Models:", reg.models)
print("Prompts:", reg.prompts)
print("System Prompts:", reg.system_prompts)

In [ ]:
GENOMIC_PIPELINE = PipelineSpec(
    name="genomic_extraction",
    model_id="gemma-3-4b-it::q4_k_m",
    system_prompt_id="mutations",
    fewshot_id="mutations",
    return_schema=GenomicBatchResult,
    inject_schema=False
)

In [ ]:
GENOMIC_PIPELINE.preview(['item1', 'item2'])

In [ ]:

reports = [
    "EGFR exon 19 deletion detected",
    "ALK negative, ROS1 negative",
]

results = await run_pipeline(
    GENOMIC_PIPELINE,
    reports,
    batch_size=5
)

In [ ]:
from pprint import pprint

from cava_llm_manager.schemas.genomic.mutations import GenomicTest


cases = [

    # perfectly valid
    {
        "genomic_marker": "EGFR",
        "test_result": "positive",
        "variant": "exon 19 deletion"
    },

    # normalisation example
    {
        "genomic_marker": "BRAF",
        "test_result": "wild type"
    },

    # another normalisation
    {
        "genomic_marker": "KRAS",
        "test_result": "mutation detected"
    },

    # phrase containing keyword
    {
        "genomic_marker": "ALK",
        "test_result": "low likelihood negative"
    },

    # unexpected value → fallback
    {
        "genomic_marker": "ROS1",
        "test_result": "probably fine"
    },

    # null case
    {
        "genomic_marker": "MET",
        "test_result": None
    },

]


print("SoftEnum behaviour demo\n")

for i, case in enumerate(cases, 1):

    test = GenomicTest.model_validate(case)

    print(f"Case {i}")
    pprint(case)

    print("Parsed result:")
    print("  marker:", test.genomic_marker)
    print("  result:", test.test_result)
    print("  variant:", test.variant)
    print("  enum_errors:", test.enum_errors)

    print("-" * 40)